# EX_16: RPS Classification — 모델 크기 극소화 (최종판)
## MobileNetV3-Small + Pruning + 회복학습 + QAT

### 파이프라인
```
Phase 1: Head 학습 (백본 고정)
Phase 2: 전체 Fine-tuning → 충분히 수렴
Phase 3: Pruning (50%, 점진적 적용)
Phase 4: 회복 학습 (strip_pruning 후 일반 Fine-tuning) ← 핵심 추가
Phase 5: QAT (INT8 양자화)
→ TFLite 변환
```
### 이전 실패 원인 및 수정
| 문제 | 원인 | 수정 |
|------|------|------|
| Pruning 후 0.33 붕괴 | 가중치 제거 후 회복 시간 없음 | **Phase 4 회복 학습 추가** |
| strip_pruning 타이밍 | 붕괴 상태로 QAT 진입 | **회복 후 QAT 적용** |
| Pruning 범위 | 전체 레이어 | **Conv/Dense만 선택적 적용** |

## Step 0. 패키지 설치
> ⚠️ 설치 후 **런타임 재시작** 필수

In [ ]:
!pip install tensorflow-model-optimization -q

## Step 1. 모듈 로딩

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import glob, cv2, pathlib, shutil, math, os
from sklearn.model_selection import train_test_split
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print('NumPy      :', np.__version__)
print('TensorFlow :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU'))

## Step 2. Google Drive 마운트

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
    colab = True
except:
    print('Local environment.')
    colab = False

## Step 3. 경로 확인 및 데이터셋 준비
- 경로 자동 검증 후 로딩

In [ ]:
files_path  = '/content/drive/MyDrive/sds/RPS_Dataset'
IMG_SIZE    = 224
NUM_CLASSES = 3
class_names = ['Rock', 'Paper', 'Scissors']

# 경로 검증
print('경로 존재:', os.path.exists(files_path))
if os.path.exists(files_path):
    print('폴더 내용:', os.listdir(files_path))
else:
    # 경로 탐색
    print('RPS 관련 경로 탐색 중...')
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if any('rps' in d.lower() for d in dirs):
            print('발견:', root, dirs)
            break
    raise FileNotFoundError(f'경로를 찾을 수 없습니다: {files_path}\n위 탐색 결과로 files_path를 수정하세요.')

print(f'\n데이터 로딩 중... (IMG_SIZE={IMG_SIZE})')
all_images, all_labels = [], []

for label in range(NUM_CLASSES):
    files = glob.glob(f'{files_path}/{label}/*.*')
    print(f'  [{label}] {class_names[label]}: {len(files)}개')
    for f in files:
        img = cv2.imread(f, cv2.IMREAD_COLOR)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        all_images.append(img)
        all_labels.append(label)

assert len(all_images) > 0, '이미지가 로딩되지 않았습니다. files_path와 폴더 구조를 확인하세요.'

all_images = np.array(all_images, dtype=np.uint8)
all_labels = np.array(all_labels)

train_data, test_data, train_labels, test_labels = train_test_split(
    all_images, all_labels,
    test_size=0.2, random_state=123, stratify=all_labels
)

print(f'\nTrain: {train_data.shape}, Test: {test_data.shape}')
print(f'클래스 분포 (train): {np.bincount(train_labels)}')
print(f'클래스 분포 (test) : {np.bincount(test_labels)}')

def mobilenet_preprocess(x):
    return (x.astype(np.float32) / 127.5) - 1.0

train_data_f  = train_data.astype(np.float32)
test_data_pre = mobilenet_preprocess(test_data)
print('데이터 준비 완료')

## Step 4. 데이터 증강

In [ ]:
img_gen_train = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=lambda x: mobilenet_preprocess(x),
    horizontal_flip=True,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    brightness_range=(0.7, 1.3),
    zoom_range=0.2,
    shear_range=10,
    fill_mode='nearest'
)

def make_gen(batch_size=32):
    return img_gen_train.flow(train_data_f, train_labels, batch_size=batch_size, seed=1)

print('ImageDataGenerator 준비 완료')

## Step 5. 베이스 모델 구성 + Phase 1/2: 일반 Fine-tuning
- Pruning 없이 먼저 충분히 수렴
- Phase 1: Head만 학습 (lr=1e-3)
- Phase 2: 전체 Fine-tuning (lr=1e-5)

In [ ]:
inputs     = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
base_model = keras.applications.MobileNetV3Small(
    weights='imagenet',
    include_top=False,
    input_tensor=inputs,
    include_preprocessing=False
)
base_model.trainable = False

x = base_model.output
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = keras.Model(inputs, outputs)

print(f'전체 파라미터: {model.count_params():,}')

# --- Phase 1: Head 학습 ---
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print('\n=== Phase 1: Head 학습 (백본 고정) ===')
history_p1 = model.fit(
    make_gen(),
    epochs=10,
    validation_data=(test_data_pre, test_labels),
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy', factor=0.5, patience=3, verbose=1, min_lr=1e-6
        )
    ]
)
_, acc_p1 = model.evaluate(test_data_pre, test_labels, verbose=0)
print(f'Phase 1 완료 - 정확도: {acc_p1:.4f}')

In [ ]:
# --- Phase 2: 전체 Fine-tuning ---
for layer in base_model.layers:
    if not isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

savedModelName_base = 'EX16_base_best.h5'

print('\n=== Phase 2: 전체 Fine-tuning (Pruning 없음) ===')
history_p2 = model.fit(
    make_gen(),
    epochs=30,
    validation_data=(test_data_pre, test_labels),
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            savedModelName_base, save_best_only=True, monitor='val_accuracy', verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy', factor=0.5, patience=4, verbose=1, min_lr=1e-8
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1
        )
    ]
)
_, acc_p2 = model.evaluate(test_data_pre, test_labels, verbose=0)
print(f'Phase 2 완료 - 정확도: {acc_p2:.4f}')
print('→ 이 정확도가 Pruning 이후 목표 기준선입니다.')

## Step 6. Phase 3: Pruning 적용
- Phase 2에서 충분히 수렴된 모델에 Pruning 적용
- `final_sparsity=0.50` (50% 제거)
- 별도 모델로 clone → 원본 가중치 유지

In [ ]:
batch_size      = 32
epochs_pruning  = 15
steps_per_epoch = math.ceil(len(train_data) / batch_size)
end_step        = steps_per_epoch * epochs_pruning

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.00,
    final_sparsity=0.50,
    begin_step=steps_per_epoch * 5,  # 5 epoch 수렴 후 Pruning 시작
    end_step=end_step
)

PRUNABLE = (
    keras.layers.Dense,
    keras.layers.Conv2D,
    keras.layers.DepthwiseConv2D
)

def apply_pruning_to_layer(layer):
    if isinstance(layer, PRUNABLE):
        return tfmot.sparsity.keras.prune_low_magnitude(
            layer, pruning_schedule=pruning_schedule
        )
    return layer

# Phase 2 최적 모델 로딩 후 Pruning 모델로 변환
model_base = keras.models.load_model(savedModelName_base)
model_for_pruning = keras.models.clone_model(
    model_base,
    clone_function=apply_pruning_to_layer
)
# 학습된 가중치를 레이어별로 복사
for src, dst in zip(model_base.layers, model_for_pruning.layers):
    try:
        dst.set_weights(src.get_weights())
    except Exception:
        pass  # Pruning wrapper는 마스크 크기 달라서 skip → 초기값 사용

model_for_pruning.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

pruned_cnt = sum(1 for l in model_for_pruning.layers if 'prune_low_magnitude' in l.name)
print(f'Pruning 적용 레이어 수: {pruned_cnt}')

print('\n=== Phase 3: Pruning (0% → 50%, 15 epochs) ===')
history_p3 = model_for_pruning.fit(
    make_gen(),
    epochs=epochs_pruning,
    validation_data=(test_data_pre, test_labels),
    callbacks=[
        tfmot.sparsity.keras.UpdatePruningStep(),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy', factor=0.5, patience=3, verbose=1, min_lr=1e-8
        )
    ]
)

_, acc_p3 = model_for_pruning.evaluate(test_data_pre, test_labels, verbose=0)
print(f'\nPruning 완료 - 정확도: {acc_p3:.4f}')
print(f'(기준선 대비: {acc_p3 - acc_p2:+.4f})')

# Pruning wrapper 제거
model_pruned = tfmot.sparsity.keras.strip_pruning(model_for_pruning)

# 실제 sparsity 확인
total_w, zero_w = 0, 0
for layer in model_pruned.layers:
    for w in layer.weights:
        if 'kernel' in w.name:
            arr = w.numpy()
            total_w += arr.size
            zero_w  += np.sum(arr == 0)
if total_w > 0:
    print(f'실제 Sparsity: {zero_w/total_w*100:.1f}%')

## Step 7. Phase 4: 회복 학습 ← 핵심 추가
- `strip_pruning` 직후 일반 모델로 추가 Fine-tuning
- Pruning으로 떨어진 정확도를 기준선 수준으로 회복
- `lr=1e-5`, EarlyStopping으로 회복되면 자동 중단

In [ ]:
model_pruned.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

savedModelName_recovered = 'EX16_pruned_recovered_best.h5'

print('=== Phase 4: 회복 학습 (Pruning 후 정확도 복원) ===')
print(f'목표: {acc_p2:.4f} 수준으로 회복')
history_p4 = model_pruned.fit(
    make_gen(),
    epochs=20,
    validation_data=(test_data_pre, test_labels),
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            savedModelName_recovered, save_best_only=True, monitor='val_accuracy', verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy', factor=0.5, patience=3, verbose=1, min_lr=1e-8
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1
        )
    ]
)

_, acc_p4 = model_pruned.evaluate(test_data_pre, test_labels, verbose=0)
print(f'\n회복 학습 완료 - 정확도: {acc_p4:.4f}')
print(f'Phase2 기준선: {acc_p2:.4f} / Pruning 후: {acc_p3:.4f} / 회복 후: {acc_p4:.4f}')

## Step 8. Phase 5: QAT 적용
- 회복된 모델에 QAT 적용
- `lr=1e-6`, 5 epochs

In [ ]:
# 회복된 최적 모델 로딩
model_recovered = keras.models.load_model(savedModelName_recovered)

def apply_quantization_to_layer(layer):
    if isinstance(layer, PRUNABLE):
        return tfmot.quantization.keras.quantize_annotate_layer(layer)
    return layer

annotated_model = keras.models.clone_model(
    model_recovered,
    clone_function=apply_quantization_to_layer
)
qat_model = tfmot.quantization.keras.quantize_apply(annotated_model)

qat_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-6),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

savedModelName_qat = 'EX16_RPS_MobileNetV3Small_QAT.h5'

print('=== Phase 5: QAT Fine-tuning ===')
history_p5 = qat_model.fit(
    make_gen(),
    epochs=5,
    validation_data=(test_data_pre, test_labels),
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            savedModelName_qat, save_best_only=True, monitor='val_accuracy', verbose=1
        )
    ]
)

_, acc_p5 = qat_model.evaluate(test_data_pre, test_labels, verbose=0)
print(f'\nQAT 완료 - 정확도: {acc_p5:.4f}')

## Step 9. 전체 학습 곡선 시각화

In [ ]:
def concat_hist(key):
    return (
        history_p1.history[key] + history_p2.history[key] +
        history_p3.history[key] + history_p4.history[key] +
        history_p5.history[key]
    )

acc_all      = concat_hist('accuracy')
val_acc_all  = concat_hist('val_accuracy')
loss_all     = concat_hist('loss')
val_loss_all = concat_hist('val_loss')

e1 = len(history_p1.history['accuracy'])
e2 = e1 + len(history_p2.history['accuracy'])
e3 = e2 + len(history_p3.history['accuracy'])
e4 = e3 + len(history_p4.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('EX_16 최종판 학습 곡선 (P1→P2→Pruning→회복→QAT)', fontsize=13)

vlines = [
    ('r',      e1, 'P1→P2'),
    ('orange', e2, 'P2→Pruning'),
    ('purple', e3, 'Pruning→회복'),
    ('green',  e4, '회복→QAT')
]

for ax, (y_train, y_val, title) in zip(axes, [
    (acc_all, val_acc_all, 'Accuracy'),
    (loss_all, val_loss_all, 'Loss')
]):
    ax.plot(y_train, 'b', label='Train')
    ax.plot(y_val,   'g', label='Validation')
    for color, x_pos, lbl in vlines:
        ax.axvline(x=x_pos - 0.5, color=color, linestyle='--', label=lbl)
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()

## Step 10. 예측 시각화

In [ ]:
from tensorflow_model_optimization.quantization.keras import quantize_scope

with quantize_scope():
    model_best = keras.models.load_model(savedModelName_qat)

y_out  = model_best.predict(test_data_pre)
y_pred = np.argmax(y_out, axis=1)

size = test_data.shape[0] // 10
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('EX_16 최종판: MobileNetV3-Small + Pruning + 회복 + QAT', fontsize=12)

for idx in range(10):
    cnt = idx * size + size // 2
    ax  = axes[idx // 5, idx % 5]
    ax.imshow(test_data[cnt])
    color = 'green' if y_pred[cnt] == test_labels[cnt] else 'red'
    ax.set_title(
        f'GT: {class_names[test_labels[cnt]]}\nPred: {class_names[y_pred[cnt]]}',
        color=color, fontsize=10
    )
    ax.axis('off')

plt.tight_layout()
plt.show()
print(f'전체 정확도: {(y_pred == test_labels).mean():.4f}')

## Step 11. TFLite 변환 및 저장

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_best)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

print(f'TFLite 모델 크기: {len(tflite_model):,} bytes ({len(tflite_model)/1024/1024:.2f} MB)')

save_dir = '/content/drive/MyDrive/files/save/' if colab else '../files/save/'
pathlib.Path(save_dir).mkdir(exist_ok=True, parents=True)

tfliteFileName = 'EX16_RPS_MobileNetV3Small_Pruning_QAT.tflite'
with open(save_dir + tfliteFileName, 'wb') as f:
    f.write(tflite_model)
shutil.copy(savedModelName_qat, save_dir + savedModelName_qat)
print(f'저장 완료: {save_dir}')

## Step 12. TFLite 정확도 평가

In [ ]:
def evaluate_tflite(model_content, test_data, test_labels):
    interpreter = tf.lite.Interpreter(model_content=model_content)
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    correct = 0
    for image, label in zip(test_data, test_labels):
        img = np.expand_dims(image.astype(np.float32), axis=0)
        interpreter.set_tensor(input_details['index'], img)
        interpreter.invoke()
        if np.argmax(interpreter.get_tensor(output_details['index'])) == label:
            correct += 1
    return correct / len(test_labels)

ex16_tflite_acc = evaluate_tflite(tflite_model, test_data_pre, test_labels)
print(f'EX_16 TFLite 정확도: {ex16_tflite_acc:.4f}')

## Step 13. 최종 비교 및 단계별 정확도 요약

In [ ]:
print('=== EX_16 단계별 정확도 변화 ===')
stages = [
    ('Phase 1 (Head 학습)',     acc_p1),
    ('Phase 2 (Fine-tuning)',   acc_p2),
    ('Phase 3 (Pruning 50%)',   acc_p3),
    ('Phase 4 (회복 학습)',     acc_p4),
    ('Phase 5 (QAT)',           acc_p5),
    ('TFLite INT8',             ex16_tflite_acc),
]
for stage, acc in stages:
    print(f'  {stage:<25}: {acc:.4f}')

print()
results = [
    ('EX_12: DenseNet121 + QAT',                   0.9927, 7_703_840),
    ('EX_14: MobileNetV2 + QAT (224px)',           0.8860, int(5.12*1024*1024)),
    ('EX_16 이전: Pruning70% (실패)',               0.3346, int(2.01*1024*1024)),
    ('EX_16 최종: Pruning50% + 회복학습 + QAT',    ex16_tflite_acc, len(tflite_model)),
]

print('=' * 68)
print(f'{"모델":<46} {"정확도":>8} {"크기(MB)":>10}')
print('-' * 68)
for name, acc, sz in results:
    print(f'{name:<46} {acc:>8.4f} {sz/1024/1024:>10.2f}')
print('=' * 68)